In [2]:
import os
import pyodbc
from struct import pack
from azure.identity import DefaultAzureCredential
import struct

SERVER = os.getenv("CHAT_SERVER_DEMO_AZURE_SQL_SERVER")
DATABASE = "master"

# Get Azure AD token
credential = DefaultAzureCredential()
token = credential.get_token("https://database.windows.net/.default")

# Correctly build UTF-16-LE access token with length prefix
exptoken = b''.join(pack('<H', ord(c)) for c in token.token)
tokenstruct = struct.pack('<I', len(exptoken)) + exptoken

# Build connection string with no user/pwd/auth
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server=tcp:{SERVER},1433;"
    f"Database={DATABASE};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
)

with pyodbc.connect(conn_str, attrs_before={1256: tokenstruct}) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT SUSER_SNAME()")
    print("Connected as:", cursor.fetchone()[0])


Connected as: live.com#andreasrasmusson1975@outlook.com
